# Hybrid SSA-N-BEATS — Multichannel Scenario

Core pipeline only: data split -> SSA decomposition & trend/seasonal split -> train trend & seasonal specialist N-BEATS models -> rolling forecast -> evaluation (MAPE, MAE, RMSE, R2).

Reusable logic lives in `src/`; this notebook only orchestrates it.

## Setup

In [ ]:
from pathlib import Path
import sys

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from pathlib import Path
import os
import sys

print("cwd      :", Path.cwd())
print("resolve  :", Path().resolve())
print("sys.path :", sys.path[:3])  

In [ ]:
from pathlib import Path

print((Path.cwd() / "../data/desember_cleaned_2024.parquet").resolve())
print((Path.cwd() / "../data/desember_cleaned_2024.parquet").exists())

In [ ]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Libs

In [ ]:
import warnings
import logging
import json
import numpy as np
import pandas as pd
import torch.nn as nn
from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler
from darts.models import NBEATSModel
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.loggers import CSVLogger

from src import (
    SSA,
    compute_w_correlation,
    auto_group_deterministic,
    split_trend_seasonal,
    rolling_forecast_multichannel,
    historical_forecast_metrics_multichannel,
    evaluate_series,
    print_evaluation_report,
)

warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)

## 1. Load data and split (in-sample / out-of-sample)

In [ ]:
df = pd.read_parquet("../data/desember_cleaned_2024.parquet")
df["DATE_TIME"] = pd.to_datetime(df["DATE_TIME"])
df = df.sort_values("DATE_TIME").set_index("DATE_TIME")
ts_full = TimeSeries.from_series(df["BEBAN"]).astype(np.float32)

split_date = pd.Timestamp("2024-10-01 00:00:00")
ts_in, ts_out = ts_full.split_before(split_date)
ts_out = ts_out.head(1488)  # 1 month at 30-minute resolution

scaler = Scaler()
ts_in_scaled = scaler.fit_transform(ts_in)
ts_out_scaled = scaler.transform(ts_out)

print(f"In-sample  : {ts_in.start_time()} to {ts_in.end_time()}")
print(f"Out-of-sample (October) : {ts_out.start_time()} to {ts_out.end_time()}")

## 2. SSA decomposition and trend/seasonal grouping (Sub-step 3.1)

In [ ]:
L_window = 336
threshold = 0.9
val_len = 1056

components, s_values = SSA(ts_in_scaled.values().flatten(), window_length=L_window)
w_corr_matrix = compute_w_correlation(components, L_window)
idx_clean = auto_group_deterministic(w_corr_matrix, threshold=threshold)
idx_trend, idx_seasonal = split_trend_seasonal(idx_clean)

trend_in_scaled = TimeSeries.from_times_and_values(
    ts_in.time_index, np.sum(components[idx_trend], axis=0)
)
seasonal_in_scaled = TimeSeries.from_times_and_values(
    ts_in.time_index, np.sum(components[idx_seasonal], axis=0)
)

train_trend = trend_in_scaled[:-val_len]
val_trend = trend_in_scaled[-val_len:]
train_seasonal = seasonal_in_scaled[:-val_len]
val_seasonal = seasonal_in_scaled[-val_len:]

print(f"Trend component uses: RC{[i + 1 for i in idx_trend]}")
print(f"Seasonal component uses {len(idx_seasonal)} RC")
print(f"Train length per channel: {len(train_trend)} points")
print(f"Validation length per channel: {len(val_trend)} points")

## 3. Train trend & seasonal specialist models (Tables 5-6: October configuration)

In [ ]:
with open("../configs/ssa_nbeats_multichannel_trend.json") as f:
    cfg_trend = json.load(f)

with open("../configs/ssa_nbeats_multichannel_seasonal.json") as f:
    cfg_seasonal = json.load(f)

logger_trend = CSVLogger("logs_skripsi", name="Multichannel_Trend_Final_October")
logger_seasonal = CSVLogger("logs_skripsi", name="Multichannel_Seasonal_Final_October")

model_trend = NBEATSModel(
    input_chunk_length=cfg_trend["input_chunk_length"],
    output_chunk_length=cfg_trend["output_chunk_length"],
    generic_architecture=True,
    num_stacks=cfg_trend["num_stacks"],
    num_blocks=cfg_trend["num_blocks"],
    num_layers=cfg_trend["num_layers"],
    layer_widths=cfg_trend["layer_widths"],
    n_epochs=cfg_trend["n_epochs"],
    batch_size=cfg_trend["batch_size"],
    random_state=cfg_trend["random_state"],
    loss_fn=nn.MSELoss(),
    optimizer_kwargs={"lr": cfg_trend["learning_rate"]},
    pl_trainer_kwargs={
        "accelerator": "gpu",
        "logger": logger_trend,
        "callbacks": [EarlyStopping(
            monitor="val_loss",
            patience=cfg_trend["early_stopping_patience"],
            min_delta=cfg_trend["min_delta"],
            mode="min",
        )],
    },
)

model_seasonal = NBEATSModel(
    input_chunk_length=cfg_seasonal["input_chunk_length"],
    output_chunk_length=cfg_seasonal["output_chunk_length"],
    generic_architecture=True,
    num_stacks=cfg_seasonal["num_stacks"],
    num_blocks=cfg_seasonal["num_blocks"],
    num_layers=cfg_seasonal["num_layers"],
    layer_widths=cfg_seasonal["layer_widths"],
    dropout=cfg_seasonal["dropout"],
    n_epochs=cfg_seasonal["n_epochs"],
    batch_size=cfg_seasonal["batch_size"],
    random_state=cfg_seasonal["random_state"],
    loss_fn=nn.MSELoss(),
    optimizer_kwargs={"lr": cfg_seasonal["learning_rate"]},
    pl_trainer_kwargs={
        "accelerator": "gpu",
        "logger": logger_seasonal,
        "callbacks": [EarlyStopping(
            monitor="val_loss",
            patience=cfg_seasonal["early_stopping_patience"],
            min_delta=cfg_seasonal["min_delta"],
            mode="min",
        )],
    },
)

print("Training trend specialist...")
model_trend.fit(series=train_trend, val_series=val_trend)

print("Training seasonal specialist...")
model_seasonal.fit(series=train_seasonal, val_series=val_seasonal)

## 4. Rolling forecast on the out-of-sample period (Table 7)

In [ ]:
final_multi_scaled = rolling_forecast_multichannel(
    model_trend,
    model_seasonal,
    ts_in_scaled,
    ts_out_scaled,
    window_length=L_window,
    threshold=threshold,
    step_size=48,
)
final_multi_mw = scaler.inverse_transform(final_multi_scaled)

## 5. Evaluation (training / validation / out-of-sample)

In [ ]:
ts_in_mw_raw = scaler.inverse_transform(ts_in_scaled)

metrics_train = historical_forecast_metrics_multichannel(
    model_trend, model_seasonal,
    train_trend, train_seasonal,
    train_trend.time_index[672], scaler, ts_in_mw_raw,
)
metrics_val = historical_forecast_metrics_multichannel(
    model_trend, model_seasonal,
    trend_in_scaled, seasonal_in_scaled,
    val_trend.start_time(), scaler, ts_in_mw_raw,
)
metrics_test = evaluate_series(ts_out, final_multi_mw)

print_evaluation_report(
    "HYBRID SSA-N-BEATS MULTICHANNEL (October 2024)",
    period_labels={
        "training": "Jan 2022 - Aug 2024",
        "validation": "Sep 2024",
        "test": "Oct 2024",
    },
    metrics_by_set={
        "training": metrics_train,
        "validation": metrics_val,
        "test": metrics_test,
    },
)